# Phase 3 — Foundation EEG Model (EEG-BERT)

This notebook fine-tunes a transformer-based foundation model (EEG-BERT) on the I-CARE EEG dataset for outcome prediction.

### Objectives
- Load preprocessed EEG segments
- Tokenize temporal sequences
- Fine-tune EEG-BERT (or compatible transformer)
- Evaluate metrics vs baseline models


## Papermill parameters

In [2]:
# Papermill parameters (overridden in automation)
preprocessed_path = '../data/icare/preprocessed'
epochs = 5
batch_size = 16
lr = 1e-4
hidden_size = 128
n_heads = 4
n_layers = 4


## Load data

In [15]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

class EEGDataset(Dataset):
    def __init__(self, path):
        self.data, self.labels = [], []
        for f in Path(path).glob('*.npy'):
            arr = np.load(f)
            if len(arr.shape) != 3:
                continue
            label = 0 if 'poor' in f.name.lower() else 1
            for segment in arr:
                self.data.append(segment)
                self.labels.append(label)
        self.data = torch.tensor(np.stack(self.data)).float()
        self.labels = torch.tensor(self.labels).long()

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

    def __len__(self):
        return len(self.data)

dataset = EEGDataset(preprocessed_path)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
print(f"Loaded {len(dataset)} segments.")


Loaded 3517 segments.


## Define EEG-BERT model

In [23]:
from transformers import BertConfig, BertModel
import torch.nn as nn
import torch

class EEG_BERT_Classifier(nn.Module):
    def __init__(self, hidden_size=128, n_heads=4, n_layers=4, num_classes=2, seq_len=1280, n_channels=19):
        super().__init__()
        
        # 1️⃣ Linear projection: EEG channels → hidden size
        self.projection = nn.Linear(n_channels, hidden_size)

        # 2️⃣ BERT configuration
        self.config = BertConfig(
            hidden_size=hidden_size,
            num_hidden_layers=n_layers,
            num_attention_heads=n_heads,
            intermediate_size=hidden_size * 2,
            max_position_embeddings=seq_len
        )

        # 3️⃣ Transformer encoder
        self.encoder = BertModel(self.config)

        # 4️⃣ Resize embeddings to match seq_len
        self.encoder.embeddings.position_embeddings = nn.Embedding(seq_len, hidden_size)
        self.encoder.embeddings.token_type_embeddings = nn.Embedding(1, hidden_size)
        self.encoder.embeddings.position_ids = torch.arange(seq_len).unsqueeze(0)

        # 5️⃣ Classification head
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x: (batch, channels, time)
        x = x.permute(0, 2, 1)            # → (batch, time, channels)
        x = self.projection(x)            # → (batch, time, hidden_size)
        outputs = self.encoder(inputs_embeds=x)
        pooled = outputs.last_hidden_state.mean(dim=1)
        return self.fc(pooled)


## Training loop

In [25]:
model = EEG_BERT_Classifier(seq_len=1280, n_channels=19).to(device)
X_sample = torch.randn(4, 19, 1280).to(device)
y_sample = model(X_sample)
print(y_sample.shape)


torch.Size([4, 2])


## Evaluation

In [26]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        preds = model(Xb)
        all_preds.extend(preds.argmax(1).cpu().numpy())
        all_true.extend(yb.cpu().numpy())

acc = accuracy_score(all_true, all_preds)
f1 = f1_score(all_true, all_preds)
print(f"✅ Accuracy: {acc:.3f} | F1: {f1:.3f}")


✅ Accuracy: 0.847 | F1: 0.917
